# Phase 2 - Problem Definition, Data Preparation, and Feature Engineering

## Prediction Objective
Given the historical crime information of a spatial unit and time-related features, predict whether that unit will be a **hotspot** in the next time block.

## Deliverables of this notebook
By the end of this notebook, we will have:

- a cleaned event-level dataset
- a District-based panel dataset
- a Community Area-based panel dataset
- hotspot labels
- lag / rolling features
- a baseline comparison between the two spatial-unit choices(1. **Police District**  2. **Community Area**)
- a recommendation on which spatial unit to use in Phase 2


## Task Design Rationale

### 1.prediction target

The prediction task is defined as:

**Given the historical crime information of a spatial unit and time-related features, predict whether that unit will be a hotspot in the next time block.**

This formulation was chosen for three reasons:

- it matches the predictive-policing setting better than describing past crime patterns only
- it is easier to evaluate as a binary classification task
- it is more suitable for later deployment and demonstration than predicting highly detailed incident-level outcomes

In particular, predicting the hotspot status of the **next time block** provides a clearer operational interpretation than predicting the current block or predicting individual incident details.

### 2.temporal granularity: 4-hour block

A 4-hour time block was chosen to balance temporal detail and data stability.

Using a very fine temporal unit such as 1 hour would make the panel much more sparse, while using only daily aggregation would remove too much short-term temporal variation.

The 4-hour design keeps the task specific enough for later demonstration, while still preserving enough signal for hotspot labeling and historical feature engineering.

### 3.spatial comparison between District and Community Area

District and Community Area were chosen because both provide **built-in area-level spatial units** in the dataset and both correspond to a **moderate spatial granularity** for crime aggregation.

They are more suitable than other available spatial fields for the current Phase 2 task:

- `beat` is too fine and would likely increase sparsity
- `ward` is less directly related to policing operations
- raw `latitude` and `longitude` are too detailed for the current hotspot design and would require a separate grid construction step

Therefore, District and Community Area were treated as the two most practical candidate spatial units for a stable and interpretable hotspot prediction workflow.

### 4.primary spatial unit: District

The structural comparison showed that Community Area was substantially more sparse than District.

Most importantly, the zero-count ratio increased from **0.0797** under District to **0.3903** under Community Area. The median crime count per unit-time dropped from **4.0** to **1.0**, and the average count dropped from **4.998** to **1.493**.

These results indicate that after moving to the finer Community Area level, many spatial-temporal units contained much weaker crime signals and a much larger share of empty units.

Given the project timeline and the need for a stable and interpretable modeling pipeline, **District** was selected as the primary spatial unit for Phase 2.

## AI Assistance Declaration

Generative AI was used in a limited supporting role in the preparation of this notebook.

Its use was limited to helping structure the workflow, refine methodological explanations, suggest code templates for preprocessing and feature engineering, and assist with interpretation of intermediate results.

All final decisions, execution of the notebook, validation of outputs, and integration into the project were completed by the student.

## 1. Notebook Setup and Raw Data Acquisition

### Why this section is needed
To keep the workflow reproducible and easy to hand off, we first:

- define consistent project paths
- create Phase 2 output folders
- retrieve the raw crime records from the official Chicago data source
- cache the raw data locally so that later steps use the same snapshot

### Output
- a standardized Phase 2 folder structure
- a cached raw dataset covering 2015-2025

In [27]:
# =========================
# 0. Imports
# =========================
import time
from io import StringIO
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

### 1.1 Configure Project Paths

In [2]:
# =========================
# 1. Project paths
# =========================
PROJECT_ROOT = Path(r"D:\AAA_nus\Sem2\IT5006\project")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "phase2"
OUTPUT_DIR = PROJECT_ROOT / "docs" / "outputs" / "phase2"
FIGURE_DIR = PROJECT_ROOT / "docs" / "figures" / "phase2"

# Create directories if they do not exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_DIR      =", RAW_DIR)
print("PROCESSED_DIR=", PROCESSED_DIR)
print("OUTPUT_DIR   =", OUTPUT_DIR)
print("FIGURE_DIR   =", FIGURE_DIR)

PROJECT_ROOT = D:\AAA_nus\Sem2\IT5006\project
RAW_DIR      = D:\AAA_nus\Sem2\IT5006\project\data\raw
PROCESSED_DIR= D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2
OUTPUT_DIR   = D:\AAA_nus\Sem2\IT5006\project\docs\outputs\phase2
FIGURE_DIR   = D:\AAA_nus\Sem2\IT5006\project\docs\figures\phase2


### 1.2 Define the Raw Data Source

#### Time range
To support the planned temporal split:

- training period: 2015-2024
- test period: 2025

we request records from:

- `2015-01-01` (inclusive)
- `2026-01-01` (exclusive)

This ensures that the full year of 2025 is included.

### Kept columns and their purposes

The selected columns are kept for the following reasons:

- `id`, `case_number`  
  Retained temporarily for record tracking, duplicate checking, and auditability.

- `date`  
  The core temporal field used to construct the prediction timeline, extract hour/day/month features, and build time blocks.

- `primary_type`, `description`  
  Not used as direct predictors in the initial hotspot model, but useful for optional descriptive analysis and future historical aggregation features.

- `location_description`  
  Useful for exploratory interpretation and possible later feature extensions, although not essential in the first baseline model.

- `arrest`, `domestic`  
  Useful for descriptive analysis and optional historical aggregate features, but not intended as direct future-time predictors.

- `beat`, `district`, `ward`, `community_area`  
  Spatial identifiers used to define candidate spatial units and support different levels of aggregation.

- `latitude`, `longitude`  
  Useful for spatial validation, mapping, and possible future grid-based analysis.

- `year`  
  Retained as a simple consistency check against parsed dates.

- `updated_on`  
  Retained only for auditing and schema reference, not for direct modeling.

In [3]:
# =========================
# 2. API configuration
# =========================
BASE_URL = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv"

START_DATE = "2015-01-01T00:00:00.000"
END_DATE   = "2026-01-01T00:00:00.000"   # exclusive, includes full year 2025

# Keep only columns potentially useful for Phase 2
SELECT_COLS = ",".join([
    "id",
    "case_number",
    "date",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "beat",
    "district",
    "ward",
    "community_area",
    "latitude",
    "longitude",
    "year",
    "updated_on"
])

RAW_DATA_PATH = RAW_DIR / "chicago_2015_2025_phase2_raw.csv"

print("BASE_URL      =", BASE_URL)
print("START_DATE    =", START_DATE)
print("END_DATE      =", END_DATE)
print("RAW_DATA_PATH =", RAW_DATA_PATH)

BASE_URL      = https://data.cityofchicago.org/resource/ijzp-q8t2.csv
START_DATE    = 2015-01-01T00:00:00.000
END_DATE      = 2026-01-01T00:00:00.000
RAW_DATA_PATH = D:\AAA_nus\Sem2\IT5006\project\data\raw\chicago_2015_2025_phase2_raw.csv


### 1.3 Build a Reusable Download Function

The raw dataset is downloaded in chunks and saved as a local cache.

#### Why chunked download
The dataset is large, so downloading in chunks is safer and more stable than requesting all rows at once.

#### Cache rule
- If the cached file already exists, load it directly.
- Otherwise, download it once and save it locally.

In [4]:
# =========================
# 3. Download helper
# =========================
def build_where_clause(start_date: str, end_date: str) -> str:
    return (
        f"date >= '{start_date}' AND date < '{end_date}' "
        "AND district IS NOT NULL "
        "AND community_area IS NOT NULL"
    )


def download_phase2_raw_csv(
    out_path: Path,
    base_url: str,
    select_cols: str,
    start_date: str,
    end_date: str,
    limit: int = 50000,
    sleep_sec: float = 0.1
) -> None:
    """
    Download Chicago crime data from the Socrata API in chunks and save to CSV.
    """
    session = requests.Session()
    offset = 0
    first_chunk = True

    while True:
        params = {
            "$select": select_cols,
            "$where": build_where_clause(start_date, end_date),
            "$order": "date ASC",
            "$limit": limit,
            "$offset": offset,
        }

        response = session.get(base_url, params=params, timeout=60)
        response.raise_for_status()

        chunk = pd.read_csv(StringIO(response.text))

        if chunk.empty:
            break

        chunk.to_csv(
            out_path,
            mode="w" if first_chunk else "a",
            index=False,
            header=first_chunk
        )

        first_chunk = False
        offset += len(chunk)
        print(f"Downloaded rows so far: {offset:,}")

        time.sleep(sleep_sec)

    print(f"Saved raw Phase 2 data to: {out_path}")

### 1.4 Load the Raw Dataset

We now load the raw dataset.

- If a local cached file already exists, we reuse it.
- Otherwise, we download the raw data from the API and save it locally.

In [5]:
# =========================
# 4. Load or download raw data
# =========================
if RAW_DATA_PATH.exists():
    print("Using cached raw dataset:", RAW_DATA_PATH)
else:
    print("No local cache found. Downloading from API...")
    download_phase2_raw_csv(
        out_path=RAW_DATA_PATH,
        base_url=BASE_URL,
        select_cols=SELECT_COLS,
        start_date=START_DATE,
        end_date=END_DATE,
        limit=50000,
        sleep_sec=0.1
    )

df_raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print("Raw dataset shape:", df_raw.shape)
df_raw.head()

No local cache found. Downloading from API...
Downloaded rows so far: 50,000
Downloaded rows so far: 100,000
Downloaded rows so far: 150,000
Downloaded rows so far: 200,000
Downloaded rows so far: 250,000
Downloaded rows so far: 300,000
Downloaded rows so far: 350,000
Downloaded rows so far: 400,000
Downloaded rows so far: 450,000
Downloaded rows so far: 500,000
Downloaded rows so far: 550,000
Downloaded rows so far: 600,000
Downloaded rows so far: 650,000
Downloaded rows so far: 700,000
Downloaded rows so far: 750,000
Downloaded rows so far: 800,000
Downloaded rows so far: 850,000
Downloaded rows so far: 900,000
Downloaded rows so far: 950,000
Downloaded rows so far: 1,000,000
Downloaded rows so far: 1,050,000
Downloaded rows so far: 1,100,000
Downloaded rows so far: 1,150,000
Downloaded rows so far: 1,200,000
Downloaded rows so far: 1,250,000
Downloaded rows so far: 1,300,000
Downloaded rows so far: 1,350,000
Downloaded rows so far: 1,400,000
Downloaded rows so far: 1,450,000
Downloa

,id,case_number,date,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,latitude,longitude,year,updated_on
0,13711023,JJ103490,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,UNAUTHORIZED VIDEOTAPING,RESIDENCE,False,True,2222,22,21.0,71,NaN,NaN,2015,2025-01-05T15:42:25.000
1,13368845,JH152261,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,DEPARTMENT STORE,False,False,111,1,34.0,32,NaN,NaN,2015,2024-02-15T15:40:52.000
2,13448318,JH247450,2015-01-01T00:00:00.000,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,RESIDENCE,False,True,1531,15,37.0,25,NaN,NaN,2015,2024-05-03T15:41:27.000
3,13307644,JG540876,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,2514,25,30.0,19,NaN,NaN,2015,2023-12-15T15:47:28.000
4,13734810,JJ132359,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,532,5,9.0,53,NaN,NaN,2015,2025-01-31T15:41:28.000


### 1.5 Quick Validation

We perform a quick validation to confirm that:

- the dataset is not empty
- the expected columns are present
- the date range covers both the training and test periods

This is only a sanity check before formal cleaning.

In [6]:
# =========================
# 5. Quick validation
# =========================
print("Columns:")
print(sorted(df_raw.columns.tolist()))

tmp_dates = pd.to_datetime(df_raw["date"], errors="coerce")

print("\nMin date:", tmp_dates.min())
print("Max date:", tmp_dates.max())

print("\nMissing ratio (top 10):")
df_raw.isna().mean().sort_values(ascending=False).head(10)

Columns:
['arrest', 'beat', 'case_number', 'community_area', 'date', 'description', 'district', 'domestic', 'id', 'latitude', 'location_description', 'longitude', 'primary_type', 'updated_on', 'ward', 'year']

Min date: 2015-01-01 00:00:00
Max date: 2025-12-31 23:58:00

Missing ratio (top 10):


latitude                0.015410
longitude               0.015410
location_description    0.004976
ward                    0.000020
primary_type            0.000000
id                      0.000000
date                    0.000000
case_number             0.000000
domestic                0.000000
arrest                  0.000000
dtype: float64

## 2. Initial Cleaning and Schema Standardization

### Why this section is needed
The raw data is suitable for storage and auditing, but not yet ready for modeling preparation.  
Before building spatial-temporal panels, we need to:

- valid timestamps
- usable spatial identifiers
- consistent data types
- non-duplicate event records

### Output

- a cleaned event-level dataset
- standardized column names
- parsed datetime variables
- a consistent schema ready for feature engineering

### 2.1 Standardize Column Names

Standardize all column names to lowercase snake_case format.

In [7]:
# =========================
# 6. Standardize column names
# =========================
df = df_raw.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print(df.columns.tolist())
df.head()

['id', 'case_number', 'date', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'latitude', 'longitude', 'year', 'updated_on']


,id,case_number,date,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,latitude,longitude,year,updated_on
0,13711023,JJ103490,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,UNAUTHORIZED VIDEOTAPING,RESIDENCE,False,True,2222,22,21.0,71,NaN,NaN,2015,2025-01-05T15:42:25.000
1,13368845,JH152261,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,DEPARTMENT STORE,False,False,111,1,34.0,32,NaN,NaN,2015,2024-02-15T15:40:52.000
2,13448318,JH247450,2015-01-01T00:00:00.000,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,RESIDENCE,False,True,1531,15,37.0,25,NaN,NaN,2015,2024-05-03T15:41:27.000
3,13307644,JG540876,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,RESIDENCE,False,False,2514,25,30.0,19,NaN,NaN,2015,2023-12-15T15:47:28.000
4,13734810,JJ132359,2015-01-01T00:00:00.000,DECEPTIVE PRACTICE,FINANCIAL IDENTITY THEFT OVER $ 300,APARTMENT,False,False,532,5,9.0,53,NaN,NaN,2015,2025-01-31T15:41:28.000


### 2.2 Parse Datetime Fields

The `date` column is the main temporal field in this project.  
We convert it into datetime format so that later steps can derive:

- event date
- hour
- weekday
- month
- time block

We also parse `updated_on` for reference, although it will not be used as a direct modeling feature.



In [8]:
# =========================
# 7. Parse datetime fields
# =========================
df["date"] = pd.to_datetime(df["date"], errors="coerce")

if "updated_on" in df.columns:
    df["updated_on"] = pd.to_datetime(df["updated_on"], errors="coerce")

print(df[["date", "updated_on"]].head())
print("\nMissing ratio in parsed datetime fields:")
print(df[["date", "updated_on"]].isna().mean())

        date          updated_on
0 2015-01-01 2025-01-05 15:42:25
1 2015-01-01 2024-02-15 15:40:52
2 2015-01-01 2024-05-03 15:41:27
3 2015-01-01 2023-12-15 15:47:28
4 2015-01-01 2025-01-31 15:41:28

Missing ratio in parsed datetime fields:
date          0.0
updated_on    0.0
dtype: float64


### 2.3 Check Spatial and Numeric Field Types

We briefly verify that the main spatial and coordinate fields are already stored in numeric format.

If the imported types are already correct, no further conversion is needed.

In [11]:
# =========================
# 8. Check spatial numeric field types
# =========================
check_cols = ["beat", "district", "ward", "community_area", "latitude", "longitude", "year"]
check_cols = [c for c in check_cols if c in df.columns]

print(df[check_cols].dtypes)

beat                int64
district            int64
ward              float64
community_area      int64
latitude          float64
longitude         float64
year                int64
dtype: object


### 2.4 Remove Duplicate Records

We remove duplicate records using a conservative subset of identifying fields.

This helps avoid double-counting incidents while keeping the deduplication rule simple and transparent.

In [ ]:
# =========================
# 9. Remove duplicates
# =========================
dup_subset = [
    "id",
    "case_number",
    "date",
    "primary_type",
    "district",
    "community_area",
    "latitude",
    "longitude"
]
dup_subset = [c for c in dup_subset if c in df.columns]

before_rows = len(df)
df = df.drop_duplicates(subset=dup_subset).copy()
after_rows = len(df)

print(f"Rows before deduplication: {before_rows:,}")
print(f"Rows after deduplication : {after_rows:,}")
print(f"Duplicates removed       : {before_rows - after_rows:,}")

Rows before deduplication: 2,756,166
Rows after deduplication : 2,756,165
Duplicates removed       : 1


### 2.5 Create Basic Temporal Fields

Before building spatial-temporal panels, we create a small set of temporal fields from the event timestamp.

These fields will later support:

- train-test splitting
- time-block aggregation
- temporal feature engineering

In [13]:
# =========================
# 10. Basic temporal fields
# =========================
df["event_date"] = df["date"].dt.floor("D")
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# 4-hour blocks: 0, 1, 2, 3, 4, 5
df["time_block"] = (df["hour"] // 4).astype(int)

df[[
    "date", "event_date", "hour", "day_of_week",
    "month", "quarter", "is_weekend", "time_block"
]].head()

,date,event_date,hour,day_of_week,month,quarter,is_weekend,time_block
0,2015-01-01,2015-01-01,0,3,1,1,0,0
1,2015-01-01,2015-01-01,0,3,1,1,0,0
2,2015-01-01,2015-01-01,0,3,1,1,0,0
3,2015-01-01,2015-01-01,0,3,1,1,0,0
4,2015-01-01,2015-01-01,0,3,1,1,0,0


### 2.6 Review the Cleaned Event-Level Dataset

We briefly confirm the size and date range of the cleaned event-level dataset before moving to panel construction.

In [16]:
# =========================
# 11. Quick review of cleaned event-level data
# =========================
print("Cleaned event-level shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nNumber of unique districts:", df["district"].nunique())
print("Number of unique community areas:", df["community_area"].nunique())

print("\nTime-block distribution:")
print(df["time_block"].value_counts().sort_index())

Cleaned event-level shape: (2756165, 23)
Date range: 2015-01-01 00:00:00 to 2025-12-31 23:58:00

Number of unique districts: 23
Number of unique community areas: 77

Time-block distribution:
time_block
0    394192
1    212053
2    463958
3    577962
4    594245
5    513755
Name: count, dtype: int64


### 2.7 Export the Cleaned Event-Level Dataset

We save the cleaned event-level dataset so that later steps can reuse it directly without repeating the same cleaning procedure.

In [17]:
# =========================
# 12. Export cleaned event-level dataset
# =========================
CLEAN_EVENT_PATH = PROCESSED_DIR / "clean_crime_events.csv"
df.to_csv(CLEAN_EVENT_PATH, index=False)

print("Saved cleaned event-level dataset to:")
print(CLEAN_EVENT_PATH)

Saved cleaned event-level dataset to:
D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2\clean_crime_events.csv


## 3. Train-Test Split and Spatial Panel Construction

### Why this section is needed
The prediction task is defined over spatial-temporal units rather than individual crime events.  
Therefore, we need to:

- split the cleaned event-level data into training and test periods
- aggregate event records into panel form
- construct two candidate panel datasets:
  - District-based
  - Community Area-based
- preserve zero-crime units explicitly for hotspot prediction

### Output of this section
By the end of this section, we will have:

- an event-level training set
- an event-level test set
- a District-based panel dataset
- a Community Area-based panel dataset
- a basic structural comparison of the two panel designs

### 3.1 Create the Train-Test Split

We first split the cleaned event-level dataset into:

- training set: 2015-2024
- test set: 2025

This temporal split matches the project design and prevents information leakage from future data.

In [18]:
# =========================
# 13. Train-test split
# =========================
train_events = df[df["event_date"].dt.year <= 2024].copy()
test_events = df[df["event_date"].dt.year == 2025].copy()

print("Train event-level shape:", train_events.shape)
print("Test event-level shape :", test_events.shape)

Train event-level shape: (2519483, 23)
Test event-level shape : (236682, 23)


### 3.2 Build the Spatial-Temporal Panel Datasets

We now convert event-level crime records into panel datasets.

Each row in the panel represents one:

- spatial unit
- event date
- 4-hour time block

We build two versions of the panel:

1. `district × event_date × time_block`
2. `community_area × event_date × time_block`

To support hotspot prediction properly, we also create the full spatial-temporal grid so that zero-crime units are explicitly included.

In [22]:
# =========================
# 14. Panel construction helper
# =========================
def build_panel(events_df, spatial_col):
    """
    Aggregate event-level records into a spatial-temporal panel.

    Each row represents one:
        spatial unit × event_date × time_block

    A full grid is created so that zero-crime units are also included.
    """
    # Aggregate observed crime counts
    grouped = (
        events_df.groupby([spatial_col, "event_date", "time_block"], as_index=False)
        .size()
        .rename(columns={"size": "crime_count"})
    )

    # Build the complete spatial-temporal grid
    spatial_values = np.sort(events_df[spatial_col].dropna().unique())
    date_values = pd.date_range(events_df["event_date"].min(), events_df["event_date"].max(), freq="D")
    block_values = np.arange(6)  # six 4-hour blocks per day

    full_index = pd.MultiIndex.from_product(
        [spatial_values, date_values, block_values],
        names=[spatial_col, "event_date", "time_block"]
    )

    panel = (
        grouped.set_index([spatial_col, "event_date", "time_block"])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    # Add temporal fields back at panel level
    panel["day_of_week"] = panel["event_date"].dt.dayofweek
    panel["month"] = panel["event_date"].dt.month
    panel["quarter"] = panel["event_date"].dt.quarter
    panel["is_weekend"] = (panel["day_of_week"] >= 5).astype(int)
    panel["year"] = panel["event_date"].dt.year

    return panel

In [23]:
# =========================
# 15. Build candidate panel datasets
# =========================
district_train = build_panel(train_events, "district")
district_test = build_panel(test_events, "district")

community_train = build_panel(train_events, "community_area")
community_test = build_panel(test_events, "community_area")

print("District train panel shape :", district_train.shape)
print("District test panel shape  :", district_test.shape)
print("Community train panel shape:", community_train.shape)
print("Community test panel shape :", community_test.shape)

District train panel shape : (504114, 9)
District test panel shape  : (50370, 9)
Community train panel shape: (1687686, 9)
Community test panel shape : (168630, 9)


### 3.3 Compare the Basic Structure of the Two Panel Designs

Before defining hotspot labels and historical features, we perform a simple structural comparison of the two panel designs.

The goal here is not to make the final decision yet, but to check whether the two spatial units differ meaningfully in terms of:

- number of spatial units
- panel size
- average crime count per unit-time
- median crime count per unit-time
- zero-crime ratio

This helps us understand potential sparsity before moving to hotspot labeling and baseline modeling.

In [ ]:
# =========================
# 16. Structural comparison of panel designs
# =========================
def summarize_panel_structure(panel, spatial_col):
    """
    Summarize the basic structure of a spatial-temporal panel.
    """
    return {
        "n_spatial_units": panel[spatial_col].nunique(),
        "n_rows": len(panel),
        "avg_count_per_unit_time": panel["crime_count"].mean(),
        "median_count_per_unit_time": panel["crime_count"].median(),
        "zero_count_ratio": (panel["crime_count"] == 0).mean(),
    }

district_summary = summarize_panel_structure(district_train, "district")
community_summary = summarize_panel_structure(community_train, "community_area")

panel_structure_comparison = pd.DataFrame(
    [district_summary, community_summary],
    index=["District", "Community Area"]
)

display(panel_structure_comparison)

,n_spatial_units,n_rows,avg_count_per_unit_time,median_count_per_unit_time,zero_count_ratio
District,23,504114,4.997844,4.0,0.079730
Community Area,77,1687686,1.492862,1.0,0.390316


#### Interpretation of the Structural Comparison

The comparison suggests that **Community Area is substantially more sparse than District**.

Most importantly, the zero-count ratio increases from **0.0797** under District to **0.3903** under Community Area. This means the share of empty spatial-temporal units rises from about **8%** to about **39%**, which would make hotspot labeling and historical feature engineering less stable.

At the same time, the median crime count per unit-time drops from **4.0** to **1.0**, and the average count drops from **4.998** to **1.493**. This indicates that after moving to the finer Community Area level, many units contain much weaker crime signals.

Given the limited project timeline and the need for a stable and interpretable prediction pipeline, **District** is the more suitable default spatial unit for Phase 2.

#### Decision

Based on the structural comparison, **District** is selected as the primary spatial unit for subsequent hotspot labeling, historical feature engineering, and baseline modeling.

### 3.4 Export the Panel Datasets

Based on the structural comparison above, the **District-based panel** is selected as the primary spatial-temporal design for Phase 2.

To keep the downstream workflow focused and efficient, the next steps will use only the District-based panel for:

- hotspot label definition
- historical feature engineering
- baseline modeling
- later model comparison

The Community Area panel is retained only as an earlier comparison reference and will not be used in the main workflow.

In [30]:
# =========================
# 17. Finalize the primary panel
# =========================
# Use District as the default panel design for the remaining workflow
train_panel = district_train.copy()
test_panel = district_test.copy()

# Save the selected primary panel datasets
train_panel.to_csv(PROCESSED_DIR / "train_panel_raw.csv", index=False)
test_panel.to_csv(PROCESSED_DIR / "test_panel_raw.csv", index=False)

# Save the structural comparison table for documentation
panel_structure_comparison.to_csv(OUTPUT_DIR / "panel_structure_comparison.csv")

print("Primary panel selected: District")
print("Saved files:")
print(PROCESSED_DIR / "train_panel_raw.csv")
print(PROCESSED_DIR / "test_panel_raw.csv")
print(OUTPUT_DIR / "panel_structure_comparison.csv")

Primary panel selected: District
Saved files:
D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2\train_panel_raw.csv
D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2\test_panel_raw.csv
D:\AAA_nus\Sem2\IT5006\project\docs\outputs\phase2\panel_structure_comparison.csv


## 4. Hotspot Label Definition and Historical Feature Engineering

### What this section does
This section defines the prediction target and creates historical features for the selected District-based panel.

### Why this section is needed
After selecting District as the primary spatial unit, we now need to:

- define a hotspot threshold using the training data only
- construct the prediction target for the **next time block**
- generate leakage-safe historical features from past crime counts
- produce feature-ready train and test datasets for downstream modeling

### Output

- a binary hotspot target
- next-block prediction labels
- lag and rolling historical features
- feature-ready train and test panel datasets

### 4.1 Define the Hotspot Threshold and the Next-Block Target

We define a hotspot using the **training panel only** so that the label rule does not depend on future data.

Then, because the prediction objective is to forecast hotspot status in the **next time block**, the final target is constructed by shifting the hotspot label forward by one block within each district.

Rows without a valid next-block label within the same split are excluded from the final modeling dataset.

In [31]:
# =========================
# 18. Define hotspot threshold and next-block target
# =========================

# Use the selected primary panel from Section 3
train_panel["dataset"] = "train"
test_panel["dataset"] = "test"

# Define hotspot threshold from training data only
hotspot_threshold = train_panel["crime_count"].quantile(0.80)

print("Hotspot threshold (80th percentile of training crime_count):", hotspot_threshold)

# Apply the same threshold to both train and test
train_panel["hotspot_current"] = (train_panel["crime_count"] >= hotspot_threshold).astype(int)
test_panel["hotspot_current"] = (test_panel["crime_count"] >= hotspot_threshold).astype(int)

print("\nCurrent hotspot ratio:")
print("Train:", train_panel["hotspot_current"].mean())
print("Test :", test_panel["hotspot_current"].mean())

Hotspot threshold (80th percentile of training crime_count): 8.0

Current hotspot ratio:
Train: 0.21771464390990927
Test : 0.1809807425054596


### 4.2 Create Leakage-Safe Historical Features

Historical features must use only information available before the prediction time.

To do this correctly, we combine the training and test panels along the full timeline, sort them by district and time, and then generate features using only past observations.

This allows the 2025 test rows to use late-2024 history, while still preventing target leakage from future periods.

In [32]:
# =========================
# 19. Combine panels and sort the full timeline
# =========================
full_panel = pd.concat([train_panel, test_panel], axis=0, ignore_index=True)

full_panel = full_panel.sort_values(
    by=["district", "event_date", "time_block"]
).reset_index(drop=True)

print("Combined full panel shape:", full_panel.shape)
full_panel.head()

Combined full panel shape: (554484, 11)


,district,event_date,time_block,crime_count,day_of_week,month,quarter,is_weekend,year,dataset,hotspot_current
0,1,2015-01-01,0,13,3,1,1,0,2015,train,1
1,1,2015-01-01,1,1,3,1,1,0,2015,train,0
2,1,2015-01-01,2,13,3,1,1,0,2015,train,1
3,1,2015-01-01,3,1,3,1,1,0,2015,train,0
4,1,2015-01-01,4,5,3,1,1,0,2015,train,0


In [ ]:
# =========================
# 20. Historical feature engineering
# =========================
def add_historical_features(panel, spatial_col="district"):
    """
    Create historical crime-count features using past values only.
    """
    panel = panel.sort_values([spatial_col, "event_date", "time_block"]).copy()
    g = panel.groupby(spatial_col)

    # Short-term lag features
    panel["count_prev_1_block"] = g["crime_count"].shift(1)
    panel["count_prev_2_blocks"] = g["crime_count"].shift(2)

    # Same time-block history
    panel["count_prev_same_block_1day"] = g["crime_count"].shift(6)    # 6 blocks per day
    panel["count_prev_same_block_7day"] = g["crime_count"].shift(42)   # 6 * 7 blocks

    # Rolling means based only on past values
    panel["rolling_mean_6_blocks"] = g["crime_count"].transform(
        lambda s: s.shift(1).rolling(window=6, min_periods=1).mean()
    )
    panel["rolling_mean_42_blocks"] = g["crime_count"].transform(
        lambda s: s.shift(1).rolling(window=42, min_periods=1).mean()
    )

    return panel

full_panel = add_historical_features(full_panel, spatial_col="district")

In [ ]:
# =========================
# 21. Construct the next-block target
# =========================
g = full_panel.groupby("district")

# Hotspot status in the next time block
full_panel["next_hotspot_current"] = g["hotspot_current"].shift(-1)

# Track whether the next row still belongs to the same split
full_panel["next_dataset"] = g["dataset"].shift(-1)

# Keep the target only if the next block stays within the same split
full_panel["target_hotspot_next_block"] = np.where(
    full_panel["dataset"] == full_panel["next_dataset"],
    full_panel["next_hotspot_current"],
    np.nan
)

full_panel[[
    "district", "event_date", "time_block", "dataset",
    "hotspot_current", "next_hotspot_current",
    "target_hotspot_next_block"
]].head(10)

,district,event_date,time_block,dataset,hotspot_current,next_hotspot_current,target_hotspot_next_block
0,1,2015-01-01,0,train,1,0.0,0.0
1,1,2015-01-01,1,train,0,1.0,1.0
2,1,2015-01-01,2,train,1,0.0,0.0
3,1,2015-01-01,3,train,0,0.0,0.0
4,1,2015-01-01,4,train,0,0.0,0.0
5,1,2015-01-01,5,train,0,0.0,0.0
6,1,2015-01-02,0,train,0,0.0,0.0
7,1,2015-01-02,1,train,0,0.0,0.0
8,1,2015-01-02,2,train,0,0.0,0.0
9,1,2015-01-02,3,train,0,1.0,1.0


### 4.3 Finalize the Feature-Ready Train and Test Datasets

We now split the full timeline back into training and test datasets.

At this stage, we keep only rows with a valid next-block target and fill the historical feature boundary gaps with zeros. These missing values occur naturally at the beginning of each district time series.

In [35]:
# =========================
# 22. Finalize feature-ready train/test datasets
# =========================
feature_cols = [
    "district",
    "event_date",
    "time_block",
    "day_of_week",
    "month",
    "quarter",
    "is_weekend",
    "crime_count",
    "hotspot_current",
    "count_prev_1_block",
    "count_prev_2_blocks",
    "count_prev_same_block_1day",
    "count_prev_same_block_7day",
    "rolling_mean_6_blocks",
    "rolling_mean_42_blocks",
    "target_hotspot_next_block",
    "dataset",
]

model_panel = full_panel[feature_cols].copy()

# Keep only rows with a valid next-block target
model_panel = model_panel[model_panel["target_hotspot_next_block"].notna()].copy()

# Fill boundary-history gaps with 0
history_cols = [
    "count_prev_1_block",
    "count_prev_2_blocks",
    "count_prev_same_block_1day",
    "count_prev_same_block_7day",
    "rolling_mean_6_blocks",
    "rolling_mean_42_blocks",
]
model_panel[history_cols] = model_panel[history_cols].fillna(0)

# Convert target to integer after filtering
model_panel["target_hotspot_next_block"] = model_panel["target_hotspot_next_block"].astype(int)

# Split back into train/test
train_featured = model_panel[model_panel["dataset"] == "train"].copy()
test_featured = model_panel[model_panel["dataset"] == "test"].copy()

print("Feature-ready train shape:", train_featured.shape)
print("Feature-ready test shape :", test_featured.shape)

print("\nTarget positive ratio:")
print("Train:", train_featured["target_hotspot_next_block"].mean())
print("Test :", test_featured["target_hotspot_next_block"].mean())

Feature-ready train shape: (504091, 17)
Feature-ready test shape : (50347, 17)

Target positive ratio:
Train: 0.21768093459315876
Test : 0.18064631457683675


In [36]:
hotspot_threshold

np.float64(8.0)

#### Interpretation of Section 4 Outputs

The hotspot threshold obtained from the training data is **8.0**, which provides a reasonably selective definition of hotspot units.

After constructing the next-block target and historical features, the resulting datasets contain **504,091** training rows and **50,347** test rows.

The positive target ratio is **0.2177** in the training set and **0.1806** in the test set. These values indicate that the target is not excessively imbalanced and remains suitable for downstream binary classification models.

In [43]:
# =========================
# 23. Save the feature-ready datasets
# =========================
train_featured.to_csv(PROCESSED_DIR / "train_panel_featured.csv", index=False)
test_featured.to_csv(PROCESSED_DIR / "test_panel_featured.csv", index=False)

print("Saved files:")
print(PROCESSED_DIR / "train_panel_featured.csv")
print(PROCESSED_DIR / "test_panel_featured.csv")

Saved files:
D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2\train_panel_featured.csv
D:\AAA_nus\Sem2\IT5006\project\data\processed\phase2\test_panel_featured.csv


#### Section Summary

This section defined the hotspot threshold using the training data only, constructed the next-block prediction target, and created historical features from past crime counts.

The resulting datasets are now ready for:

- baseline model training
- model comparison
- later interpretation of temporal crime patterns

## 5. Handoff Notes for Model Implementation

### Output usage
The next team member should use:

- `train_panel_featured.csv` as the training dataset
- `test_panel_featured.csv` as the test dataset

These two files already contain:

- the selected District-based spatial unit
- the 4-hour time-block structure
- the next-block prediction target
- the historical crime-count features needed for baseline modeling

The prediction target is:

- `target_hotspot_next_block`

The recommended starting feature set is:

- `district`
- `time_block`
- `day_of_week`
- `month`
- `quarter`
- `is_weekend`
- `crime_count`
- `hotspot_current`
- `count_prev_1_block`
- `count_prev_2_blocks`
- `count_prev_same_block_1day`
- `count_prev_same_block_7day`
- `rolling_mean_6_blocks`
- `rolling_mean_42_blocks`

### Settings That Must Remain Consistent

The following settings should remain unchanged in all downstream modeling and evaluation steps:

- spatial unit: **District**
- temporal unit: **4-hour block**
- train-test split: **2015-2024 for training, 2025 for testing**
- prediction target: **target_hotspot_next_block**
- hotspot threshold definition: **80th percentile of training crime_count**
- historical feature logic: **past-only information, no future leakage**

These settings define the modeling task itself.  
Changing them would create a different prediction problem and would make later model comparisons inconsistent.

### Suggested File Organization for the Next Team Member

A simple and practical structure is recommended:

- processed datasets remain in: `data/processed/phase2/`
- model outputs remain in: `docs/outputs/phase2/`
- evaluation figures remain in: `docs/figures/phase2/`
- model notebooks remain in: `notebooks/phase2/`

For example, the next team member may create:

- `notebooks/phase2/02_model_implementation.ipynb`
- `docs/outputs/phase2/model_metrics_summary.csv`
- `docs/figures/phase2/roc_curve_logistic.png`

This structure is not mandatory, but following it will reduce confusion and make final integration easier.
